# Grain Elevator Pipeline — Example

Run the full or partial pipeline: **OCR → Extraction → CSV**.

Make sure `.env` is configured (see `.env.example`).

In [ ]:
# auto reload modules when they change (for development)
%load_ext autoreload
%autoreload 2
from run_pipeline import run_pipeline
import anthropic
from extract_elevators import recover_batch, jsonl_to_csv, submit_batch, list_batches
from pathlib import Path

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [6]:
# -- Paths --
YEAR = "2013"                    # label for output filenames
IMAGES_DIR = f"G:/.shortcut-targets-by-id/1a2KJDruB4ByO9oJKAZto52DF4dvwN3Y3/grain_and_milling/scans_img_segmented/Elevator/{YEAR}"       # directory with scanned page images
OUTPUT_DIR = f"G:/.shortcut-targets-by-id/1a2KJDruB4ByO9oJKAZto52DF4dvwN3Y3/grain_and_milling/OCR_google_api"           # where all output files go


# -- Select which steps to run --
do_ocr = False                    # Step 1: Google Document AI OCR
do_extraction = True             # Step 2: Claude structured extraction
do_csv = True                    # Step 3: JSONL to flat CSV

backend = "claude"              # Ollama or  Claude (see README for setup instructions)
model = "claude-sonnet-4-6"          # Ollama model name, e.g. "llama3.1:8b" or "claude-haiku-4-5" or "claude-sonnet-4-6"
batching = True                 # Whether to use batch processing for extraction (if supported by the model)

## Run Pipeline

In [ ]:
run_pipeline(
    images_dir=IMAGES_DIR,
    output_dir=OUTPUT_DIR,
    year=YEAR,
    do_ocr=do_ocr,
    do_extraction=do_extraction,
    do_csv=do_csv,
    backend=backend,
    model=model,
    batching=False,
)

2026-03-09 10:33:38,424 [INFO] Extraction: G:\.shortcut-targets-by-id\1a2KJDruB4ByO9oJKAZto52DF4dvwN3Y3\grain_and_milling\OCR_google_api\2013_ocr.csv → G:\.shortcut-targets-by-id\1a2KJDruB4ByO9oJKAZto52DF4dvwN3Y3\grain_and_milling\OCR_google_api\2013.jsonl (backend=claude, batching=True)
2026-03-09 10:33:38,448 [INFO] Total rows: 924  |  status=success: 923
2026-03-09 10:33:38,449 [INFO] Using backend: claude  |  model: claude-sonnet-4-6
2026-03-09 10:33:38,461 [INFO] Already processed: 0 rows — will skip these.
2026-03-09 10:33:38,505 [INFO] Merging continuation 2013_1_18.png into 2013_1_17.png
2026-03-09 10:33:38,506 [INFO] Merging continuation 2013_4_1.png into 2013_4_0.png
2026-03-09 10:33:38,511 [INFO] Merging continuation 2013_21_27.png into 2013_21_26.png
2026-03-09 10:33:38,512 [INFO] Merging continuation 2013_25_8.png into 2013_25_7.png
2026-03-09 10:33:38,512 [INFO] Merging continuation 2013_26_8.png into 2013_26_7.png
2026-03-09 10:33:38,513 [INFO] Merging continuation 2013_

## Batch Processing with Claude

In [ ]:
client = anthropic.Anthropic()

# List recent batches
for batch in client.messages.batches.list(limit=10):
    print(f"{batch.id}: {batch.processing_status} | created: {batch.created_at}")

2026-03-09 10:46:59,301 [INFO] HTTP Request: GET https://api.anthropic.com/v1/messages/batches?limit=10 "HTTP/1.1 200 OK"


msgbatch_01EB79Yt4rRspTzmMyTzbyQ1: ended | created: 2026-03-09 13:33:37.744186+00:00
msgbatch_01VeTXC6VwD8JWm656hp43Ri: ended | created: 2026-03-09 13:33:10.980720+00:00


In [ ]:
# 1. Submit batch (returns immediately)
batch_id = submit_batch(
    f"{OUTPUT_DIR}/{YEAR}_ocr.csv",
    model=model
)
print(f"Submitted: {batch_id}")

# 2. Check status later
for b in list_batches():
    print(f"{b['id']}: {b['status']} ({b['succeeded']}/{b['processing']} done)")

# 3. Recover results when complete
successes, errors = recover_batch(
    batch_id,
    f"{OUTPUT_DIR}/{YEAR}.jsonl"
)
print(f"Recovered {successes} entries, {errors} errors")

# 4. Convert to CSV
jsonl_to_csv(f"{OUTPUT_DIR}/{YEAR}.jsonl", f"{OUTPUT_DIR}/{YEAR}.csv")

In [ ]:
BATCH_ID = "msgbatch_01EB79Yt4rRspTzmMyTzbyQ1"  # <-- paste your batch ID from cell 5

# Recover
successes, errors = recover_batch(BATCH_ID, f"{OUTPUT_DIR}/{YEAR}.jsonl")

# Convert to CSV
jsonl_to_csv(f"{OUTPUT_DIR}/{YEAR}.jsonl", f"{OUTPUT_DIR}/{YEAR}.csv")

2026-03-09 10:42:56,047 [INFO] HTTP Request: GET https://api.anthropic.com/v1/messages/batches/msgbatch_01EB79Yt4rRspTzmMyTzbyQ1 "HTTP/1.1 200 OK"
2026-03-09 10:42:56,049 [INFO] Retrieving results from batch msgbatch_01EB79Yt4rRspTzmMyTzbyQ1...
2026-03-09 10:42:56,240 [INFO] HTTP Request: GET https://api.anthropic.com/v1/messages/batches/msgbatch_01EB79Yt4rRspTzmMyTzbyQ1 "HTTP/1.1 200 OK"
2026-03-09 10:42:56,593 [INFO] HTTP Request: GET https://api.anthropic.com/v1/messages/batches/msgbatch_01EB79Yt4rRspTzmMyTzbyQ1/results "HTTP/1.1 200 OK"
2026-03-09 10:42:56,705 [ERROR] Parse error for 2013_7_21.png: Expecting property name enclosed in double quotes: line 1 column 471 (char 470)
2026-03-09 10:42:56,778 [ERROR] Parse error for 2013_21_26+2013_21_27.png: Expecting value: line 1 column 1 (char 0)
2026-03-09 10:42:56,879 [INFO] Recovered 922 entries, 2 errors
2026-03-09 10:42:56,880 [INFO] Output: G:\.shortcut-targets-by-id\1a2KJDruB4ByO9oJKAZto52DF4dvwN3Y3\grain_and_milling\OCR_google_a

## Inspect Results

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path(OUTPUT_DIR) / f"{YEAR}.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"{len(df)} entries extracted")
    df.head()
else:
    print(f"No output yet at {csv_path} -- run the pipeline first.")